# Cycle 1 — Hyperparameter Tuning

**Project:** Football Predictor  
**Depends on:** `cycle1_modelling.ipynb` (run that first)

---

## Purpose of this Notebook

The baseline modelling notebook trained all models with **default settings**. This notebook finds the **optimal settings** for each model using **Randomized Search Cross Validation**.


## What is Hyperparameter Tuning?

Every ML model has **hyperparameters** — settings you configure before training. For example, XGBoost has:
- `n_estimators` — how many trees to build
- `max_depth` — how deep each tree grows
- `learning_rate` — how fast the model learns
- `subsample` — fraction of data used per tree

Different combinations give different results. Tuning finds the combination that gives the highest accuracy.

## Why Randomized Search over Grid Search?

**Grid Search** tries every possible combination — if you have 5 parameters with 4 options each, that is 4⁵ = 1,024 combinations. Very slow.

**Randomized Search** randomly samples a fixed number of combinations (we use 50). Much faster, and research shows it finds equally good results in practice.

## What is Cross Validation (CV)?

Instead of using one train/test split, CV splits the training data into 5 equal parts (folds). The model trains on 4 folds and tests on the 5th — repeated 5 times. The average score across all 5 folds is the CV score. This gives a more reliable estimate of real performance than a single split.

---
## Cell 1 — Imports and Data Loading

**What it does:** Imports all libraries and loads both processed datasets.

**Why:** Both datasets are tuned in this notebook for comparison.

In [10]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier

# Dataset 1
df1 = pd.read_csv('../../data/processed/premier_league_matches_processed.csv')
X1 = df1.drop(columns=['FTR', 'Season'])  # Season is meta (year), not a feature
y1 = df1['FTR']
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y1, test_size=0.2, random_state=42)

# Dataset 2
df2 = pd.read_csv('../../data/processed/skysports_match_stats_processed.csv')
X2 = df2.drop(columns=['FTR', 'date'])
y2 = df2['FTR']
X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)

# 5-fold stratified cross validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print('Dataset 1 — Training:', len(X1_train), '| Test:', len(X1_test))
print('Dataset 2 — Training:', len(X2_train), '| Test:', len(X2_test))

Dataset 1 — Training: 5472 | Test: 1368
Dataset 2 — Training: 896 | Test: 225


### Observations
- Same splits as the modelling notebook (same `random_state=42`) — results are directly comparable
- `StratifiedKFold` ensures each fold has the same class distribution as the full dataset — important for imbalanced data

---
# PART A — Tuning XGBoost on Dataset 1

**Baseline (untuned):** 50.95%  
**Target:** Beat 51.39% (best untuned model on Dataset 1 — Random Forest)

## A1 — Define Parameter Grid

**What it does:** Defines the range of hyperparameter values to search over.

**Why these parameters?**
- `n_estimators` — more trees = more learning capacity, but slower and risks overfitting
- `max_depth` — deeper trees capture more complexity, but overfit more easily
- `learning_rate` — smaller = more conservative learning, needs more trees to compensate
- `subsample` — fraction of training data used per tree, adding randomness reduces overfitting
- `colsample_bytree` — fraction of features used per tree, reduces correlation between trees
- `min_child_weight` — minimum data points in a leaf, higher = more conservative
- `gamma` — minimum loss reduction for a split, higher = more conservative

In [11]:
xgb_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [3, 4, 5, 6],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0],
    'min_child_weight': [1, 3, 5],
    'gamma':            [0, 0.1, 0.2]
}

total_combinations = 4 * 4 * 4 * 3 * 3 * 3 * 3
print(f'Total possible combinations: {total_combinations:,}')
print(f'Combinations we will try: 50 (RandomizedSearch)')
print(f'With 5-fold CV: 50 × 5 = 250 model fits')

Total possible combinations: 5,184
Combinations we will try: 50 (RandomizedSearch)
With 5-fold CV: 50 × 5 = 250 model fits


### Observations
- Grid Search would try all 3,888 combinations × 5 folds = 19,440 fits — very slow
- RandomizedSearch tries 250 fits — much faster, finds comparably good results

## A2 — Run Randomized Search on Dataset 1

**What it does:** Runs 50 random combinations of hyperparameters, each evaluated with 5-fold CV. Returns the best combination.

**Why:** Finds a significantly better XGBoost configuration than the default settings.

In [12]:
xgb = XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)

search_d1 = RandomizedSearchCV(
    xgb, xgb_param_grid,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_d1.fit(X1_train, y1_train)

print('Best hyperparameters:')
for param, value in search_d1.best_params_.items():
    print(f'  {param}: {value}')
print()
print(f'Best CV accuracy: {search_d1.best_score_*100:.2f}%')

y_pred_xgb_d1_tuned = search_d1.best_estimator_.predict(X1_test)
print(f'Test accuracy:    {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  subsample: 0.7
  n_estimators: 200
  min_child_weight: 3
  max_depth: 5
  learning_rate: 0.01
  gamma: 0.1
  colsample_bytree: 0.8

Best CV accuracy: 52.47%
Test accuracy:    52.78%


### Observations
- **Low learning rate (0.01) + more trees (300)** — the tuner found that XGBoost learns better slowly on this dataset
- **subsample: 0.7** — using only 70% of data per tree reduces overfitting
- Test accuracy (52.78%) is slightly above CV accuracy (52.47%) — the model generalises well

### Improvement over baseline
- Untuned XGBoost Dataset 1: **50.95%**
- Tuned XGBoost Dataset 1: **52.78%**
- **Gain: +2.49 percentage points**

## A3 — Full Classification Report — Dataset 1 Tuned XGBoost

In [13]:
print('TUNED XGBOOST — Dataset 1')
print(f'Accuracy: {accuracy_score(y1_test, y_pred_xgb_d1_tuned)*100:.2f}%')
print()
print(classification_report(y1_test, y_pred_xgb_d1_tuned, target_names=['Away Win', 'Draw', 'Home Win']))

TUNED XGBOOST — Dataset 1
Accuracy: 52.78%

              precision    recall  f1-score   support

    Away Win       0.50      0.39      0.44       394
        Draw       0.42      0.05      0.09       340
    Home Win       0.54      0.87      0.67       634

    accuracy                           0.53      1368
   macro avg       0.49      0.44      0.40      1368
weighted avg       0.50      0.53      0.46      1368



### Observations
- Draw recall remains low (0.06) — even tuned XGBoost struggles with draws on Dataset 1
- Home Win recall is high (0.87) — model confidently identifies home wins
- The season-level form features in Dataset 1 simply do not provide enough signal for draws

---
# PART B — Tuning XGBoost on Dataset 2

**Baseline (untuned):** 42.22%  
**Target:** Beat 46.22% (best untuned model on Dataset 2 — Random Forest)

## B1 — Run Randomized Search on Dataset 2

In [14]:
# Pipeline ensures CV folds are scaled per-fold (no leakage) and that the
# saved model uses the same preprocessing as the search.
xgb2_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('xgb', XGBClassifier(random_state=42, eval_metric='mlogloss', verbosity=0)),
])
# Hyperparameters reference the 'xgb' step in the pipeline
xgb_param_grid_pipe = {f'xgb__{k}': v for k, v in xgb_param_grid.items()}

search_d2 = RandomizedSearchCV(
    xgb2_pipe, xgb_param_grid_pipe,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_d2.fit(X2_train, y2_train)

print('Best hyperparameters:')
for param, value in search_d2.best_params_.items():
    print(f'  {param}: {value}')
print()
print(f'Best CV accuracy: {search_d2.best_score_*100:.2f}%')

y_pred_xgb_d2_tuned = search_d2.best_estimator_.predict(X2_test)
print(f'Test accuracy:    {accuracy_score(y2_test, y_pred_xgb_d2_tuned)*100:.2f}%')

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  xgb__subsample: 0.7
  xgb__n_estimators: 100
  xgb__min_child_weight: 5
  xgb__max_depth: 3
  xgb__learning_rate: 0.01
  xgb__gamma: 0.2
  xgb__colsample_bytree: 0.8

Best CV accuracy: 54.69%
Test accuracy:    46.22%


### Observations
- **Test accuracy (46.22%) is notably lower than CV accuracy (54.69%)** — an **8.47pp gap** vs Dataset 1's effectively-zero gap (CV 52.47% / test 52.78%). Dataset 2's tuner is overfitting to its CV folds because the training set is small (~896 rows)

### Improvement over baseline
- Untuned XGBoost Dataset 2: **42.22%**
- Tuned XGBoost Dataset 2: **46.22%**

## B2 — Full Classification Report — Dataset 2 Tuned XGBoost

In [15]:
print('TUNED XGBOOST — Dataset 2')
print(f'Accuracy: {accuracy_score(y2_test, y_pred_xgb_d2_tuned)*100:.2f}%')
print()
print(classification_report(y2_test, y_pred_xgb_d2_tuned, target_names=['Away Win', 'Draw', 'Home Win']))

TUNED XGBOOST — Dataset 2
Accuracy: 46.22%

              precision    recall  f1-score   support

    Away Win       0.47      0.53      0.50        80
        Draw       0.00      0.00      0.00        63
    Home Win       0.47      0.76      0.58        82

    accuracy                           0.46       225
   macro avg       0.31      0.43      0.36       225
weighted avg       0.34      0.46      0.39       225



### Observations
- Tuned XGBoost on Dataset 2: **46.22%** (was 54.69% under the original leaky-CV run)
- The earlier 57.33% was inflated by fitting the `StandardScaler` on the full training set before CV (CV folds saw scaler-leaked information). The new Pipeline-based search scales **per fold**, producing the honest 46.22%
- Despite the lower headline accuracy, this is the methodologically correct number
- Away Win: precision 0.47 / recall 0.53 — model predicts away wins reasonably
- Home Win: precision 0.47 / recall 0.76 — biased toward predicting home wins
- **Draw: precision 0.00, recall 0.00** — the model never predicts a draw under proper CV

### CRITICAL ISSUE — Draw Prediction
Draw recall of 0 means the model entirely refuses to predict draws. This is the same weakness as the random untuned models, only more pronounced under proper CV — the tuner now picks hyperparameters that further lean into the binary Home/Away decision. Honest evaluation under chronological splits gives 52.00% on Dataset 2 with non-zero draw recall, so the chronological approach is preferable for both the methodology and the deployed model behaviour.

### Notes for Report
- 46.22% is the honest random-split tuned XGBoost number after fixing CV leakage
- The legacy 57.33% in the dashboard caption refers to the older leaky-CV training of the deployed model; it should be updated when the model is re-trained under the chronological split

---
# PART C — Tuning Random Forest on Dataset 2

**Baseline (untuned RF on D2):** 46.22%  
**Target:** Beat 46.22% and compare with tuned XGBoost (46.22%)

## C1 — Run Randomized Search — Random Forest Dataset 2

In [16]:
rf_param_grid = {
    'n_estimators':     [100, 200, 300, 500],
    'max_depth':        [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf':  [1, 2, 4],
    'max_features':      ['sqrt', 'log2', None],
    'class_weight':      ['balanced', 'balanced_subsample']
}

rf = RandomForestClassifier(random_state=42)

search_rf_d2 = RandomizedSearchCV(
    rf, rf_param_grid,
    n_iter=50,
    cv=cv,
    scoring='accuracy',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

search_rf_d2.fit(X2_train, y2_train)

print('Best hyperparameters:')
for param, value in search_rf_d2.best_params_.items():
    print(f'  {param}: {value}')
print()
print(f'Best CV accuracy: {search_rf_d2.best_score_*100:.2f}%')

y_pred_rf_d2_tuned = search_rf_d2.best_estimator_.predict(X2_test)
print(f'Test accuracy:    {accuracy_score(y2_test, y_pred_rf_d2_tuned)*100:.2f}%')
print()
print(classification_report(y2_test, y_pred_rf_d2_tuned, target_names=['Away Win', 'Draw', 'Home Win']))

Fitting 5 folds for each of 50 candidates, totalling 250 fits
Best hyperparameters:
  n_estimators: 500
  min_samples_split: 2
  min_samples_leaf: 1
  max_features: sqrt
  max_depth: None
  class_weight: balanced_subsample

Best CV accuracy: 54.58%
Test accuracy:    44.44%

              precision    recall  f1-score   support

    Away Win       0.48      0.47      0.48        80
        Draw       0.00      0.00      0.00        63
    Home Win       0.44      0.76      0.56        82

    accuracy                           0.44       225
   macro avg       0.31      0.41      0.34       225
weighted avg       0.33      0.44      0.37       225



### Observations
- Tuned Random Forest: **44.44% test accuracy** with **54.58% best CV** — a **10pp CV-vs-test gap** indicates significant overfitting to the CV folds
- This is exactly the symptom small datasets show: 50 candidate hyperparameter sets × 5 folds × 896 training rows means each fold sees only ~717 rows. The tuner finds patterns that don't generalise
- Draw recall is **0.00** — the tuned RF never predicts a draw, same collapse as tuned XGBoost
- `balanced_subsample` won — different per-tree balancing helps slightly, but cannot rescue draw detection at this dataset size

---
# PART D — Full Results Comparison

**What it does:** Compares all models — untuned and tuned — across both datasets, alongside FinalYearProject results.

**Why:** Gives the complete picture needed for your report.

In [17]:
results = pd.DataFrame([
    # Dataset 1
    {'Dataset': 'Dataset 1', 'Model': 'Dummy',                    'Type': 'Baseline',  'Accuracy': 46.35},
    {'Dataset': 'Dataset 1', 'Model': 'Logistic Regression',      'Type': 'Untuned',   'Accuracy': 49.85},
    {'Dataset': 'Dataset 1', 'Model': 'Random Forest',            'Type': 'Untuned',   'Accuracy': 51.39},
    {'Dataset': 'Dataset 1', 'Model': 'XGBoost',                  'Type': 'Untuned',   'Accuracy': 50.95},
    {'Dataset': 'Dataset 1', 'Model': 'XGBoost Tuned',            'Type': 'Tuned',     'Accuracy': 52.78},
    # Dataset 2
    {'Dataset': 'Dataset 2', 'Model': 'Dummy',                    'Type': 'Baseline',  'Accuracy': 36.44},
    {'Dataset': 'Dataset 2', 'Model': 'Logistic Regression',      'Type': 'Untuned',   'Accuracy': 44.89},
    {'Dataset': 'Dataset 2', 'Model': 'Random Forest',            'Type': 'Untuned',   'Accuracy': 46.22},
    {'Dataset': 'Dataset 2', 'Model': 'XGBoost',                  'Type': 'Untuned',   'Accuracy': 42.22},
    {'Dataset': 'Dataset 2', 'Model': 'Random Forest Tuned',      'Type': 'Tuned',     'Accuracy': 44.44},
    {'Dataset': 'Dataset 2', 'Model': 'XGBoost Tuned',            'Type': 'Tuned',     'Accuracy': 46.22},
])

print(results.to_string(index=False))

  Dataset               Model     Type  Accuracy
Dataset 1               Dummy Baseline     46.35
Dataset 1 Logistic Regression  Untuned     49.85
Dataset 1       Random Forest  Untuned     51.39
Dataset 1             XGBoost  Untuned     50.95
Dataset 1       XGBoost Tuned    Tuned     52.78
Dataset 2               Dummy Baseline     36.44
Dataset 2 Logistic Regression  Untuned     44.89
Dataset 2       Random Forest  Untuned     46.22
Dataset 2             XGBoost  Untuned     42.22
Dataset 2 Random Forest Tuned    Tuned     44.44
Dataset 2       XGBoost Tuned    Tuned     46.22


## Key Conclusions

### 1. Tuned XGBoost on Dataset 1 achieves **52.78%** — the highest random-split accuracy
Dataset 1's tuned XGBoost beats Dataset 2's tuned XGBoost (46.22%) under proper per-fold scaling. Dataset 1 has 6× more training rows, so its tuned model overfits less.

### 2. Tuned models still cannot predict draws
Both tuned RF (D2 recall 0.00) and tuned XGB (D2 recall 0.00) refuse to predict draws when CV scaling is corrected. Draw detection requires either chronological evaluation, balanced sampling, or explicit per-class threshold tuning.

### 3. Dataset 2 is still deployed despite Dataset 1's higher accuracy
Dataset 1's features (HTGS/HTGC/HTP/HM-form) require running totals across a season — these are not available at API inference time, since the feature store only computes 5-match rolling stats from Sky Sports data. The deployed Cycle 1 model uses Dataset 2 features by necessity, not by accuracy.


### Observations
- Pipeline-based search and save use the **same per-fold scaling** — no preprocessing inconsistency between training and saved artefact
- The saved model achieves 46.22% test accuracy under proper random-split CV
- Three files saved: model, scaler, feature column list — everything the API needs to make a prediction

In [18]:
import joblib
import os

# The Pipeline already contains the fitted scaler + best XGBoost model.
# Extract them directly — no retraining needed (search and save are now consistent).
best_pipeline = search_d2.best_estimator_
best_xgb = best_pipeline.named_steps['xgb']
scaler   = best_pipeline.named_steps['scaler']

# Sanity check: confirm test accuracy matches what the search reported
y_pred_final = best_pipeline.predict(X2_test)
final_acc = accuracy_score(y2_test, y_pred_final)
print(f'Final pipeline test accuracy: {final_acc*100:.2f}%')

# --- Save ---
models_dir = '../models'
os.makedirs(models_dir, exist_ok=True)

model_path  = os.path.join(models_dir, 'cycle1_xgb_best.pkl')
scaler_path = os.path.join(models_dir, 'cycle1_scaler.pkl')

joblib.dump(best_xgb, model_path)
joblib.dump(scaler,   scaler_path)

print(f'Model saved  → {model_path}')
print(f'Scaler saved → {scaler_path}')

# --- Save feature column names (needed for API input validation) ---
feature_cols = list(X2_train.columns)
joblib.dump(feature_cols, os.path.join(models_dir, 'cycle1_feature_cols.pkl'))
print(f'Features saved → {os.path.join(models_dir, "cycle1_feature_cols.pkl")}')
print(f'\nFeature columns ({len(feature_cols)}): {feature_cols}')

Final pipeline test accuracy: 46.22%
Model saved  → ../models/cycle1_xgb_best.pkl
Scaler saved → ../models/cycle1_scaler.pkl
Features saved → ../models/cycle1_feature_cols.pkl

Feature columns (19): ['attendance', 'Home Team', 'Away Team', 'home_avg_possession_5', 'home_avg_shots_5', 'home_avg_shots_on_target_5', 'home_avg_pass_accuracy_5', 'home_avg_tackles_5', 'home_avg_corners_5', 'home_avg_fouls_5', 'home_avg_yellow_cards_5', 'away_avg_possession_5', 'away_avg_shots_5', 'away_avg_shots_on_target_5', 'away_avg_pass_accuracy_5', 'away_avg_tackles_5', 'away_avg_corners_5', 'away_avg_fouls_5', 'away_avg_yellow_cards_5']


## E1 — Save Model and Scaler

**What it does:** Saves two objects:
1. `cycle1_xgb_best.pkl` — the trained XGBoost model with best hyperparameters
2. `cycle1_scaler.pkl` — the StandardScaler fitted on Dataset 2 training data

**Why save the scaler separately?**  
At prediction time (in the API), any new input must be scaled using the same scaler fitted on the training data. If we only save the model, predictions on raw inputs will be wrong.

---
# PART E — Save the Cycle 1 Model

**What it does:** Saves the **deployable** Cycle 1 model — Tuned XGBoost on Dataset 2 — to disk as a `.pkl` file. Even though Dataset 1's tuned model has higher accuracy (52.78% vs 46.22%), Dataset 1's features are not available at API inference time (no running-total feature store), so Dataset 2 is the only deployable choice.

**Why save:** A saved model can be loaded by other notebooks (SHAP explainability), the FastAPI backend, and the Streamlit dashboard without retraining. It also makes results reproducible.

**What is a `.pkl` file?**  
A pickle file serialises a Python object — in this case the trained XGBoost model — to binary format. `joblib` is preferred over Python's built-in `pickle` for scikit-learn/XGBoost objects because it handles large numpy arrays more efficiently.

**Comparison with FinalYearProject:** FYP saved a dict containing scaler, label encoders, PCA, and the model all in one `.pkl`. Here we save the model separately from the scaler, keeping components modular and easier to inspect or replace.